In [ ]:
import pandas as pd

from keras.layers import Flatten , Dense  , Embedding , Input , Concatenate
from keras.optimizers import Adam
from keras.losses import BinaryCrossentropy
from keras.models import Model
from tensorflow.random import set_seed

In [ ]:
x_train = pd.read_csv('./datas/Out_Stage2/x_train')
x_test  = pd.read_csv('./datas/Out_Stage2/x_test' )
x_valid = pd.read_csv('./datas/Out_Stage2/x_valid')
y_train = pd.read_csv('./datas/Out_Stage2/y_train')
y_test  = pd.read_csv('./datas/Out_Stage2/y_test' )
y_valid = pd.read_csv('./datas/Out_Stage2/y_valid')
x_train_scaled = x_train .copy()
x_test_scaled = x_test  .copy()
x_valid_scaled = x_valid .copy()
y_train_scaled = y_train .copy()
y_test_scaled = y_test  .copy()
y_valid_scaled = y_valid .copy()

In [ ]:
x_train.info()

In [ ]:
del_cols = ['MARRIAGE_0' ,
'MARRIAGE_1' ,
'MARRIAGE_2' ,
'MARRIAGE_3' ,
'EDUCATION_0',
'EDUCATION_1',
'EDUCATION_2',
'EDUCATION_3',
'EDUCATION_4',
'EDUCATION_5',
'EDUCATION_6',
'SEX_1',      
'SEX_2'      ]

embedding_cols = ['MARRIAGE', 'EDUCATION','SEX']

numeric_cols = ['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6','AGE','BILL_AMT1','BILL_AMT2',
                'BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6','PAY_AMT1','PAY_AMT2','PAY_AMT3','PAY_AMT4','PAY_AMT5','PAY_AMT6']



In [ ]:
x_train_scaled = x_train.drop(del_cols , axis=1)
x_test_scaled = x_test.drop(del_cols , axis=1)
x_valid_scaled = x_valid.drop(del_cols , axis=1)
x_train_scaled['SEX'] = x_train['SEX']-1
x_test_scaled['SEX'] = x_test['SEX']-1
x_valid_scaled['SEX'] = x_valid['SEX']-1

In [ ]:
input_marriage = Input(shape=(1,) , name='marriage')
embeding_marriage = Embedding(4,4 // 2 ,name = 'marriage_emb')(input_marriage)
layer_marriage = Flatten()(embeding_marriage)

input_education = Input(shape=(1,) , name='education')
embeding_education = Embedding(7,7 // 2 ,name = 'education_emb')(input_education)
layer_education = Flatten()(embeding_education)

input_sex = Input(shape=(1,) , name='sex')
embeding_sex = Embedding(2,2 // 2 ,name = 'sex_emb')(input_sex)
layer_sex = Flatten()(embeding_sex)

In [ ]:
numeric_input = Input(shape=(19,))
layer_numeric = Dense(32, activation='relu')(numeric_input)

In [ ]:
set_seed(42)

concat = Concatenate()([layer_marriage,layer_sex,layer_education ,layer_numeric])


hidden = Dense(64, activation='relu')
hidden2 = Dense(64, activation='relu')
output = Dense(1,activation='sigmoid')

layer_main = hidden(concat)
layer_main = hidden2(layer_main)
layer_output = output(layer_main)

model_mainnormal = Model(inputs=[input_sex, input_marriage , input_education , numeric_input] , outputs =[layer_output])

model_mainnormal.compile(optimizer=Adam(learning_rate=1e-3) , loss=BinaryCrossentropy() ,metrics=['accuracy'])

In [ ]:
history_mainnormal_model = model_mainnormal.fit(
    [x_train_scaled['SEX'],x_train_scaled['MARRIAGE'],x_train_scaled['EDUCATION'],x_train_scaled[numeric_cols]]
     , y_train_scaled  , epochs=5 ,
       validation_data=(
           [x_valid_scaled['SEX'],x_valid_scaled['MARRIAGE'],x_valid_scaled['EDUCATION'],x_valid_scaled[numeric_cols]]
           ,y_valid_scaled) , )

In [ ]:
x_train_scaled .to_csv('./datas/Out_Stage3/x_train',index=False)
x_test_scaled .to_csv('./datas/Out_Stage3/x_test ',index=False)
x_valid_scaled .to_csv('./datas/Out_Stage3/x_valid',index=False)
y_train_scaled .to_csv('./datas/Out_Stage3/y_train',index=False)
y_test_scaled .to_csv('./datas/Out_Stage3/y_test ',index=False)
y_valid_scaled .to_csv('./datas/Out_Stage3/y_valid',index=False)